# Cinematic Bokeh Demo

Interactive playground. Drop an image into `assets/input/`, set the path below, then run all cells.

Click on the displayed RGB image to pick a focus point (in the cell that supports it), or pass coords manually.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from src.pipeline import BokehPipeline, BokehConfig
from src.utils import load_image, colorize_depth, overlay_mask
from src.evaluate import summarize

In [ ]:
INPUT_PATH = '../assets/input/portrait.jpg'  # <-- change me
rgb = load_image(INPUT_PATH)
plt.figure(figsize=(8, 6)); plt.imshow(rgb); plt.title('Input'); plt.axis('off'); plt.show()

In [ ]:
cfg = BokehConfig(
    depth_variant='vits',
    max_blur_px=30.0,
    falloff=1.3,
    kernel_shape='hexagonal',
    n_layers=7,
    highlight_boost=1.7,
)
pipeline = BokehPipeline(cfg)

In [ ]:
result = pipeline.run(rgb, focus_point=None)
print(f'Focus depth = {result.focus_depth:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0,0].imshow(result.rgb);                        axes[0,0].set_title('Input')
axes[0,1].imshow(colorize_depth(result.depth_raw));  axes[0,1].set_title('Depth Anything V2 (raw)')
axes[0,2].imshow(colorize_depth(result.depth_refined)); axes[0,2].set_title('Refined depth (ours)')
axes[1,0].imshow(overlay_mask(result.rgb, result.mask));    axes[1,0].set_title('Subject mask')
axes[1,1].imshow(result.bokeh_baseline);             axes[1,1].set_title('Bokeh baseline')
axes[1,2].imshow(result.bokeh_refined);              axes[1,2].set_title('Bokeh refined (ours)')
for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
import json
metrics = summarize(result)
print(json.dumps(metrics, indent=2))

## Sweep: focus pulling

Render the same image with multiple focus distances to demonstrate user-controllable focus.

In [ ]:
from src.bokeh import render_bokeh
depths_to_focus = [0.2, 0.45, 0.7, 0.9]
fig, axes = plt.subplots(1, len(depths_to_focus), figsize=(20, 5))
for ax, fd in zip(axes, depths_to_focus):
    out = render_bokeh(result.rgb, result.depth_refined, focus_depth=fd,
                       max_blur_px=cfg.max_blur_px, falloff=cfg.falloff,
                       kernel_shape=cfg.kernel_shape, n_layers=cfg.n_layers,
                       highlight_boost=cfg.highlight_boost)
    ax.imshow(out); ax.set_title(f'focus={fd:.2f}'); ax.axis('off')
plt.tight_layout(); plt.show()